## Prompt Pitfalls

- Vagueness: Occurs when the prompt lacks precise criteria, context, or clear boundaries. The model is forced to make broad assumptions, leading to generic filler text, inconsistent depth, and uncontrolled output lengths.

- Ambiguity: Occurs when directives, phrasing, or target keys can be interpreted in multiple valid ways. This results in structural drift and fluctuating output logic across identical runs.

- Hallucination: Occurs when the model invents plausible-sounding facts to fulfill an ungrounded or underspecified prompt. The model attempts to complete patterns without authoritative source boundaries or explicit fallback rules.

## Baseline vs. Enhanced Prompting & Pydantic Contracts

In [1]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

raw_profile = """
Rohan Sharma is a Lead Data Engineer based in Bengaluru with experience since 2017.
He specializes in building scalable batch and streaming pipelines using Apache Spark, PySpark, 
Kafka, Databricks, and Snowflake. He has architected Medallion data lakehouses on AWS using S3 
and Delta Lake. Rohan holds the AWS Certified Data Analytics - Specialty certification.
"""

In [2]:
# Vague Prompt: No criteria, length target, or output formatting rules
vague_prompt = f"Write something about this profile:\n\n{raw_profile}"

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=vague_prompt,
    config=types.GenerateContentConfig(temperature=0.8)  # Highlight variance
)

print("=== [Pitfall 1: Vagueness Output] ===")
print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


=== [Pitfall 1: Vagueness Output] ===
Here are a few ways to describe Rohan Sharma's profile, depending on the context:

---

**1. Concise Summary (Good for a quick overview or internal note):**

> Rohan Sharma is a Bengaluru-based Lead Data Engineer with experience since 2017. He specializes in building scalable batch and streaming data pipelines using Apache Spark, PySpark, Kafka, Databricks, and Snowflake. Rohan has also architected Medallion data lakehouses on AWS (S3, Delta Lake) and holds the AWS Certified Data Analytics - Specialty certification.

---

**2. LinkedIn "About" Section (More descriptive and engaging):**

> As a Lead Data Engineer based in Bengaluru with experience since 2017, I excel at transforming complex data challenges into scalable, high-performance solutions. My expertise lies in architecting and implementing end-to-end batch and streaming data pipelines using a robust tech stack including Apache Spark, PySpark, Kafka, Databricks, and Snowflake. I have a prove

In [ ]:
# Ambiguous Prompt: Undefined keys and loose schema requirements
ambiguous_prompt = f"""Please give the candidate details as JSON:
- Name
- Experience (seniority)
- Skills

Profile:
{raw_profile}"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=ambiguous_prompt,
    config=types.GenerateContentConfig(temperature=1.0)
)

print("=== [Pitfall 2: Ambiguity Output] ===")
print(response.text)

=== [Pitfall 2: Ambiguity Output] ===
```json
{
  "Name": "Rohan Sharma",
  "Experience": {
    "Seniority": "Lead Data Engineer",
    "Years_Since": "2017 (approximately 6-7 years of experience as of late 2023/early 2024)"
  },
  "Skills": [
    "Apache Spark",
    "PySpark",
    "Kafka",
    "Databricks",
    "Snowflake",
    "AWS",
    "S3",
    "Delta Lake",
    "Medallion Data Lakehouse Architecture",
    "Batch Processing",
    "Streaming Pipelines",
    "AWS Certified Data Analytics - Specialty"
  ]
}
```


In [ ]:
import json
try:
    json_response = json.loads(response.text) #code that may raise exception
except json.JSONDecodeError as e:
    print(f"JSON parsing error !:{e}") #runs when exception occurs
else:
    print("JSON parsed correctly!") #runs only when 'try' succeeds
finally:
    print("JSON parsing test finished") # Runs always
# print(json_response)

JSON parsing error !:Expecting value: line 1 column 1 (char 0)


In [18]:
# Hallucination Prompt: Demands ungrounded facts without providing fallback instructions
hallucination_prompt = f"""Extract the following fields from the profile:
- Candidate Name and age
- Highest Academic Degree & Graduation Year
- Apex experience
- Primary Cloud Architecture
- Opensource tools exposure

Data:
{raw_profile}"""

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=hallucination_prompt,
    config=types.GenerateContentConfig(temperature=1.2)
)

print("=== [Pitfall 3: Hallucination Output] ===")
print(response.text)

=== [Pitfall 3: Hallucination Output] ===
Here are the extracted fields:

- **Candidate Name and age:** Rohan Sharma (Age: N/A)
- **Highest Academic Degree & Graduation Year:** N/A
- **Apex experience:** Since 2017
- **Primary Cloud Architecture:** AWS
- **Opensource tools exposure:** Apache Spark, PySpark, Kafka, Delta Lake


## Enhanced Prompt

In [ ]:
system_instruction = (
        "You are an enterprise talent data ingestion engine. Extract structured engineer"
        "profiles strictly according to the format instructions. Do not invent missing facts. Fill N/A for missing information"
    )

prompt = f"""### INSTRUCTION
Extract the candidate profile to strict JSON format.

### INPUT DATA
\"\"\"
{raw_profile}
\"\"\"

### RULES
1. Calculate years_of_experience assuming current year is 2026.
2. primary_skills must be an array of strings.
3. certifications must be an array of strings.
4. Suppress all conversational markdown outside the raw JSON object.

### OUTPUT INDICATOR
{{
"full_name": "string",
"current_role": "string",
"years_of_experience": int,
"primary_skills": ["string"],
"certifications": ["string"]
}}"""

response = client.models.generate_content(
    model="gemini-3.7-flash",
    contents=prompt,
    config=types.GenerateContentConfig(
        system_instruction=system_instruction,
        temperature=0.0,  # Greedy decoding for consistency
        response_mime_type = "text/plain"
    )
)
print("\n--- [Enhanced Contract JSON Output] ---")
print(response.text)


--- [Enhanced Contract JSON Output] ---
{
  "full_name": "Rohan Sharma",
  "current_role": "Lead Data Engineer",
  "years_of_experience": 9,
  "primary_skills": [
    "Apache Spark",
    "PySpark",
    "Kafka",
    "Databricks",
    "Snowflake",
    "AWS",
    "S3",
    "Delta Lake"
  ],
  "certifications": [
    "AWS Certified Data Analytics - Specialty"
  ]
}


In [33]:
import json
try:
    data_dict = json.loads(response.text)
    print("valid JSON")
except json.JSONDecodeError as e:
    print("Json parsing error")


valid JSON


## Pydantic 
- Data validation, parsing, and settings management library for Python powered by standard type annotations. 
- Built on a core written in Rust (Pydantic v2)
- It converts untrusted, semi-structured, or raw inputs into strongly typed, validated Python data models.

### Core Pillars of Pydantic
**Type Safety & Type Casting:** Automatically coerces incoming data types (e.g., converting the string "2026" into an int 2026).  
**Declarative Constraints (Field):** Enforces numerical boundaries, string limits, regex patterns, and default values directly within class definitions (e.g., Field(ge=0, min_length=1)).  
**Custom Validation Logic (@field_validator / @model_validator):** Enables custom sanitation, normalization (such as title-casing or date formatting), and cross-field logic.  
**Nested Data Modeling:** Allows composing relational hierarchies using nested BaseModel classes (e.g., a list of sub-objects).  
**Runtime Error Handling:** Surfaces detailed ValidationError objects identifying the exact offending key and validation failure.

In [ ]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, field_validator

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# 1. Single model definition
# Book inherits from Pydantic core class BaseModel
class Book(BaseModel):
    title: str = Field(description="Exact title of the book")
    author: list[str] = Field(min_length=1, max_length=10, description="Name of the author(s)")
    edition: str = Field(default="N/A",description="Edition information (e.g., '10th Edition')")
    publisher: str = Field(description="Publishing company or academic press")
    publication_year: int = Field(ge=1900, le=2026, description="Year of publication")


# 2. Query with search grounding & top-level list schema
def get_textbooks_json(count: int, subject: str) -> str:
    prompt = f"Search the web and list the top {count} authoritative textbooks for '{subject}'."

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())],
            # response_mime_type="application/json",
            response_schema=list[Book],  # Top-level array schema
            temperature=0.0,
        ),
    )
    return response.text

if __name__ == "__main__":
    json_output = get_textbooks_json(count=3, subject="Engineering Mathematics")
    
    # Direct raw JSON string output
    print(json_output)

Here are three of the top authoritative textbooks for 'Engineering Mathematics':

*   **"Advanced Engineering Mathematics" by Erwin Kreyszig** This classic textbook is widely recognized as a comprehensive and authoritative resource in the field. It is highly regarded for building concepts and providing in-depth material.
*   **"Higher Engineering Mathematics" by B.S. Grewal** This book is frequently recommended as a comprehensive solution for first-year B.Tech students, covering almost all topics in Engineering Mathematics 1 & 2, and is often cited for exam preparation.
*   **"Engineering Mathematics" by K.A. Stroud and Dexter J. Booth** Known for its structured approach and focus on practical applications, this textbook is a strong alternative, especially for those who might find Kreyszig too advanced initially. It covers a wide range of topics from foundational arithmetic to complex numbers, differential equations, and statistics.


In [49]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError, field_validator

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

# 1. Pydantic Contract
class Book(BaseModel):
    title: str = Field(description="Exact title of the book")
    author: list[str] = Field(min_length=1, max_length=10, description="List of author names")
    edition: str = Field(default="N/A", description="Edition information")
    publisher: str = Field(description="Publishing company")
    publication_year: int = Field(ge=1900, le=2026, description="Year of publication")
    @field_validator("title")
    @classmethod
    def sanitize_title(cls, v: str) -> str:
        return v.strip().title()

def get_grounded_textbooks(count: int, subject: str) -> str:
    # --- Stage 1: Search & Verify Facts ---
    search_prompt = f"Search the web and find the top {count} authoritative textbooks for '{subject}'. Include title, author(s), publisher, latest edition, and publication year."
    
    grounded_res = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=search_prompt,
        config=types.GenerateContentConfig(
            tools=[types.Tool(google_search=types.GoogleSearch())],
            temperature=0.0
        )
    )
    raw_research = grounded_res.text

    # --- Stage 2: Strict JSON Schema Extraction ---
    extract_prompt = f"""Extract the textbook metadata from the research notes below into the required schema.

    Research Notes:
    \"\"\"
    {raw_research}
    \"\"\""""

    structured_res = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=extract_prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json",
            response_schema=list[Book],
            temperature=0.0
        )
    )
    return structured_res.text

if __name__ == "__main__":
    json_output = get_grounded_textbooks(count=3, subject="Engineering Mathematics")
    print(json_output)

[{"title":"Advanced Engineering Mathematics","author":["Erwin Kreyszig"],"edition":"10th Edition","publisher":"Wiley","publication_year":2011},{"title":"Higher Engineering Mathematics","author":["B.S. Grewal"],"edition":"45th Edition","publisher":"Khanna Publishers","publication_year":2024},{"title":"Engineering Mathematics","author":["K.A. Stroud","Dexter J. Booth"],"edition":"8th Edition","publisher":"Red Globe Press","publication_year":2020}]


#### BaseModel is the foundational base class in Pydantic. 
- Inheriting from BaseModel gives your class several capabilities:  
Automatic Type Validation & Coercion: If an API returns "2026" (a string) for a field annotated as int, Pydantic parses it into the integer 2026.  
- Built-in Deserialization Methods: Provides class methods like .model_validate_json() to parse JSON strings and instantiate typed Python objects.  
- Serialization Utilities: Provides methods like .model_dump() (to convert to a standard Python dict) and .model_dump_json() (to export as a clean JSON string).
- Runtime Error Raising: If required data is missing or incompatible, it raises a structured ValidationError rather than crashing silently.